In [ ]:
from tqdm import tqdm
import numpy as np
from PIL import Image
from rfdetr import RFDETRBase
from roboflow.core import dataset as roboflow_dataset

In [ ]:
dataset = roboflow_dataset.Dataset(name="custom_cell_dataset", version="0.1", model_format="coco", 
                location="/home/cellareye/Cellanome/dl-mehdi/Mask RCNN/data/all_datasets_0p25_suspension_10x_bf_yolo")

In [ ]:
model = RFDETRBase()
history = []

def callback2(data):
    history.append(data)

model.callbacks["on_fit_epoch_end"].append(callback2)
# for efficient training, make sure total batch_size as batch_size * grad_accum_steps is equal to 16
# resolution should be divisible by 14 (because of Dinov2 backbone) and also 4
model.train(dataset_dir=dataset.location, 
            epochs=8, 
            batch_size=4, 
            grad_accum_steps=4, 
            lr=1e-4, 
            resolution=672, 
            output_dir='rf_detr_checkpoints')

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.DataFrame(history)

plt.figure(figsize=(12, 8))

plt.plot(
    df['epoch'],
    df['train_loss'],
    label='Training Loss',
    marker='o',
    linestyle='-'
)

plt.plot(
    df['epoch'],
    df['test_loss'],
    label='Validation Loss',
    marker='o',
    linestyle='--'
)

plt.title('Train/Validation Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
df = pd.DataFrame(history)

df['avg_precision'] = df['test_coco_eval_bbox'].apply(lambda arr: arr[0])
df['avg_recall'] = df['test_coco_eval_bbox'].apply(lambda arr: arr[6])

plt.figure(figsize=(12, 8))
plt.plot(
    df['epoch'],
    df['avg_precision'],
    marker='o',
    linestyle='-'
)
plt.title('AP (IoU=0.50:0.95, area=all, maxDets=100) over Epochs')
plt.xlabel('Epoch')
plt.ylabel('AP')
plt.grid(True)
plt.show()


plt.figure(figsize=(12, 8))
plt.plot(
    df['epoch'],
    df['avg_recall'],
    marker='o',
    linestyle='-'
)
plt.title('AR (IoU=0.50:0.95, area=all, maxDets=1) over Epochs')
plt.xlabel('Epoch')
plt.ylabel('AR')
plt.grid(True)
plt.show()

In [ ]:
# needed for inference
import supervision as sv
from supervision.metrics import MeanAveragePrecision

ds_valid = sv.DetectionDataset.from_coco(
    images_directory_path=f"{dataset.location}/test",
    annotations_path=f"{dataset.location}/test/_annotations.coco.json",
)

In [ ]:
model = RFDETRBase(pretrain_weights="rf_detr_checkpoints/checkpoint_best_total.pth")

targets = []
predictions = []

for path, image, annotations in tqdm(ds_valid):
    detections = model.predict(image, threshold=0.001)

    targets.append(annotations)
    predictions.append(detections)
    

In [ ]:
map_metric = MeanAveragePrecision()
map_result = map_metric.update(predictions, targets).compute()

map_result.plot()

In [ ]:
import sys
sys.path.append("../utils")
from pairing_utils import pair_gts_dets_bbox
from typing import Dict, List, Final, Tuple
CONFIDENCES: Final[List[float]] = [i / 200 for i in range(40, 101, 5)]

def evaluate_pr_curve(
    targets:  List[sv.detection.core.Detections],
    predictions: List[sv.detection.core.Detections],
    confidences: List[float] = CONFIDENCES,
    class_ids_of_interest: List[int] = [0, 1, 2, 3],
    min_iou: float = 0.5,
) -> Dict[float, Tuple[float, float]]:
    """
    A function to evaluate precision/recall values for different threshold values
    Args:
        targets: List of ground truth annotations for each image, each element should be a supervision.detection.core.Detections 
            object with members xyxy and class_id
        predictions: List of detections for each image, each element should be a supervision.detection.core.Detections object with
            members xyxy, confidence and class_id
        confidences: List of confidence threshold values to calculate precision/recall at.
        class_ids_of_interest (list or 1-D np.ndarray): A list of class IDs for labels to consider in
            precision/recall evaluation.
        min_iou (float): Minimum IoU between the mask of a ground truth and that of a detection
            to declare a detection correct.
    Returns
        A dictionary with keys as values in the passed confidences and values (precision, recall) tuple for
            the given confidence threshold.
    """

    num_true_positives = {conf: 0 for conf in confidences}
    num_false_positives = {conf: 0 for conf in confidences}
    num_false_negatives = {conf: 0 for conf in confidences}

    print(f"Calculating precision/recall for {len(predictions)} images")
    detected_class_ids = set()
    target_class_ids = set()
    
    
    for idx, detection in enumerate(predictions):
        # ground truth annotations for the image
        target = targets[idx]
        # predictions
        boxes = detection.xyxy
        labels = detection.class_id
        scores = detection.confidence

        detected_class_ids = detected_class_ids.union(set(labels))
        target_class_ids = target_class_ids.union(set(target.class_id))
       
        for class_id in class_ids_of_interest:
            # filter the detections and ground truths for the given label
            class_idxs = np.where(target.class_id == class_id)[0]
            gt_boxes = target.xyxy[class_idxs].astype(int)
            
            class_idxs = np.where(labels == class_id)[0]
            det_boxes = boxes[class_idxs].astype(int)
            det_scores = scores[class_idxs]

            for conf in confidences:
                # filter the low confidence detections
                det_boxes_with_confidence = det_boxes[det_scores >= conf, :]
                # pair
                paired_idx, unpaired_gts, unpaired_dets = pair_gts_dets_bbox(
                    gt_boxes,
                    det_boxes_with_confidence,
                    min_iou,
                )

                num_true_positives[conf] += len(paired_idx)
                num_false_positives[conf] += len(unpaired_dets)
                num_false_negatives[conf] += len(unpaired_gts)

        if (idx + 1) % 100 == 0:
            print(f"Completed {idx + 1} images out of {len(predictions)}")

    p_r_dict: Dict[float, Tuple[float, float, float]] = {}
    for conf in confidences:
        p: float = num_true_positives[conf] / (num_true_positives[conf] + num_false_positives[conf] + 1e-30)
        r: float = num_true_positives[conf] / (num_true_positives[conf] + num_false_negatives[conf] + 1e-30)
        f_1: float = 2 * p * r / (p + r)
        p_r_dict[conf] = (p, r, f_1)

    print(f"All detected class IDs: {detected_class_ids}")
    print(f"All target class IDs: {target_class_ids}")
    return p_r_dict

In [ ]:
p_r_dict = evaluate_pr_curve(targets=targets, predictions=predictions)

In [ ]:
p_r_dict

In [ ]:
len(ds_valid)

In [ ]:
img_path, img, anns = ds_valid[11421]

In [ ]:
image = Image.fromarray(img)
detections = model.predict(img, threshold=0.5)

color = sv.ColorPalette.from_hex([
    "#ffff00", "#ff9b00", "#ff8080", "#ff66b2", "#ff66ff", "#b266ff",
    "#9999ff", "#3399ff", "#66ffff", "#33ff99", "#66ff66", "#99ff00"
])
text_scale = sv.calculate_optimal_text_scale(resolution_wh=image.size)
thickness = sv.calculate_optimal_line_thickness(resolution_wh=image.size)

bbox_annotator = sv.BoxAnnotator(color=color, thickness=thickness)
label_annotator = sv.LabelAnnotator(
    color=color,
    text_color=sv.Color.BLACK,
    text_scale=text_scale,
    smart_position=True
)

labels = [
    f"{ds_valid.classes[class_id]} {confidence:.2f}"
    for class_id, confidence
    in zip(detections.class_id, detections.confidence)
]

annotated_image = image.copy()
annotated_image = bbox_annotator.annotate(annotated_image, detections)
annotated_image = label_annotator.annotate(annotated_image, detections, labels)
annotated_image

In [ ]:
img.shape